# Lab 4 - Klotski

In [6]:
from collections import deque
import heapq

## Read the problems

In [7]:
def read_problems(filename):
    problems = []
    board = None
    for line in open(filename):
        line = line.strip()
        words = line.split()
        if line.startswith("problem"):
            problem = {"name": words[1]}
            problems.append(problem)
        elif line.startswith("size"):
            problem["rows"] = int(words[1])
            problem["cols"] = int(words[2])
        elif line.startswith("exit"):
            problem["exit"] = [int(x) for x in words[2:]]
        elif line.startswith("target"):
            problem["target"] = words[1]
        elif line.startswith("====="):
            if board is None:
                board = []
            else:
                problem["board"] = board
                board = None
        elif board is not None and line != "":
            board.append(words)
    return problems


problems = read_problems("klotski_problems.txt")
print(problems[0])

{'name': 'p01', 'rows': 3, 'cols': 3, 'exit': [1], 'target': 'A', 'board': [['A', '5', '1'], ['6', '4', '2'], ['3', '.', '.']]}


## Pieces

In [8]:
# for every piece: its size (height, width)
# the state: the top-left corner of every piece
def find_pieces(problem):
    names = []
    sizes = []
    start = []
    for r in range(problem["rows"]):
        for c in range(problem["cols"]):
            name = problem["board"][r][c]
            if name != "." and name not in names:
                # first time we see this piece: this is its top-left corner
                h = 1
                while r + h < problem["rows"] and problem["board"][r + h][c] == name:
                    h += 1
                w = 1
                while c + w < problem["cols"] and problem["board"][r][c + w] == name:
                    w += 1
                names.append(name)
                sizes.append((h, w))
                start.append((r, c))
    problem["names"] = names
    problem["sizes"] = sizes
    problem["target_index"] = names.index(problem["target"])
    problem["start"] = tuple(start)


for problem in problems:
    find_pieces(problem)

p = problems[0]
print(p["names"])
print(p["sizes"])
print(p["start"])

['A', '5', '1', '6', '4', '2', '3']
[(1, 1), (1, 1), (1, 1), (1, 1), (1, 1), (1, 1), (1, 1)]
((0, 0), (0, 1), (0, 2), (1, 0), (1, 1), (1, 2), (2, 0))


## Board

In [9]:
# build the board from a state
def make_board(problem, state):
    board = [["." for c in range(problem["cols"])] for r in range(problem["rows"])]
    for i in range(len(state)):
        r, c = state[i]
        h, w = problem["sizes"][i]
        for x in range(h):
            for y in range(w):
                board[r + x][c + y] = problem["names"][i]
    return board


def print_board(problem, state):
    for row in make_board(problem, state):
        print(" ".join(row))
    print()


print_board(p, p["start"])

A 5 1
6 4 2
3 . .



## Goal and moves

In [10]:
# the target must be on the last row, over the exit columns
def is_goal(problem, state):
    board = make_board(problem, state)
    last_row = board[problem["rows"] - 1]
    for col in problem["exit"]:
        if last_row[col] != problem["target"]:
            return False
    return True


# move one piece by one cell: up, down, left, right
def get_moves(problem, state):
    board = make_board(problem, state)
    new_states = []
    for i in range(len(state)):
        r, c = state[i]
        h, w = problem["sizes"][i]
        name = problem["names"][i]
        for dr, dc in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
            new_r = r + dr
            new_c = c + dc
            # outside the board?
            if new_r < 0 or new_c < 0 or new_r + h > problem["rows"] or new_c + w > problem["cols"]:
                continue
            # the new cells must be empty or part of the same piece
            ok = True
            for x in range(h):
                for y in range(w):
                    cell = board[new_r + x][new_c + y]
                    if cell != "." and cell != name:
                        ok = False
            if ok:
                new_state = list(state)
                new_state[i] = (new_r, new_c)
                new_states.append(tuple(new_state))
    return new_states


for s in get_moves(p, p["start"]):
    print_board(p, s)

A 5 1
6 . 2
3 4 .

A 5 1
6 4 .
3 . 2

A 5 1
6 4 2
. 3 .



## Same board, different names

If two pieces of the same size swap places, the board is the same.
So to remember the boards we have already seen, we use the size of the pieces instead of their names.

In [11]:
def key(problem, state):
    others = []
    for i in range(len(state)):
        if i != problem["target_index"]:
            others.append((problem["sizes"][i], state[i]))
    others.sort()
    return (state[problem["target_index"]], tuple(others))

In [12]:
# from the goal go back to the start
def get_path(problem, parent, state):
    path = [state]
    while parent[key(problem, state)] is not None:
        state = parent[key(problem, state)]
        path.append(state)
    path.reverse()
    return path

## Solution 1: BFS

In [13]:
def bfs(problem):
    start = problem["start"]
    parent = {key(problem, start): None}
    queue = deque([start])
    while len(queue) > 0:
        state = queue.popleft()
        if is_goal(problem, state):
            return get_path(problem, parent, state), len(parent)
        for new_state in get_moves(problem, state):
            if key(problem, new_state) not in parent:
                parent[key(problem, new_state)] = state
                queue.append(new_state)
    return None, len(parent)


path, seen = bfs(p)
print("moves:", len(path) - 1, "  seen:", seen)
for state in path:
    print_board(p, state)

moves: 8   seen: 141
A 5 1
6 4 2
3 . .

A 5 1
6 . 2
3 4 .

A . 1
6 5 2
3 4 .

. A 1
6 5 2
3 4 .

6 A 1
. 5 2
3 4 .

6 A 1
5 . 2
3 4 .

6 . 1
5 A 2
3 4 .

6 . 1
5 A 2
3 . 4

6 . 1
5 . 2
3 A 4



## Solution 2: A*

Priority = moves done + heuristic.
Heuristic = distance of the target from the exit.

In [14]:
def heuristic(problem, state):
    r, c = state[problem["target_index"]]
    h, w = problem["sizes"][problem["target_index"]]
    down = problem["rows"] - (r + h)
    side = abs(c - problem["exit"][0])
    return down + side


def a_star(problem):
    start = problem["start"]
    parent = {key(problem, start): None}
    cost = {key(problem, start): 0}
    queue = [(heuristic(problem, start), start)]
    while len(queue) > 0:
        priority, state = heapq.heappop(queue)
        if is_goal(problem, state):
            return get_path(problem, parent, state), len(parent)
        for new_state in get_moves(problem, state):
            k = key(problem, new_state)
            new_cost = cost[key(problem, state)] + 1
            if k not in cost or new_cost < cost[k]:
                cost[k] = new_cost
                parent[k] = state
                heapq.heappush(queue, (new_cost + heuristic(problem, new_state), new_state))
    return None, len(parent)


path, seen = a_star(p)
print("moves:", len(path) - 1, "  seen:", seen)

moves: 8   seen: 121


## All problems

In [15]:
for problem in problems:
    path1, seen1 = bfs(problem)
    path2, seen2 = a_star(problem)
    if path1 is None:
        print(problem["name"], " no solution")
    else:
        print(problem["name"], " BFS:", len(path1) - 1, "moves,", seen1, "seen   A*:", len(path2) - 1, "moves,", seen2, "seen")

p01  BFS: 8 moves, 141 seen   A*: 8 moves, 121 seen
p02  BFS: 18 moves, 266 seen   A*: 18 moves, 259 seen
p03  BFS: 45 moves, 2153 seen   A*: 45 moves, 2129 seen
p04  BFS: 116 moves, 24098 seen   A*: 116 moves, 23934 seen
p05  BFS: 179 moves, 40469 seen   A*: 179 moves, 39215 seen
p06  no solution
